# 🏆 Notebook 05 — Evaluación Final y Demo del Pipeline

**Objetivo:** Reportar los resultados definitivos del modelo y demostrar el pipeline end-to-end.

- Métricas finales sobre el test set completo (25,250 imágenes)
- Matriz de confusión y análisis de errores
- Accuracy vs. umbral de confianza
- **Grad-CAM**: visualizar qué ve el modelo
- Demo del pipeline: imagen → categoría → nutrición
- Análisis de error calórico (MAE y MAPE)


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.config import DATA_DIR, WEIGHTS_DIR, DEVICE, SEED, BATCH_SIZE, FOOD101_CLASSES
from src.model import load_model
from src.pipeline import FoodVisionPipeline
from src.nutrition import get_nutrition
from src.transforms import get_val_transform, get_inference_transform
from src.utils import plot_confusion_matrix, per_class_accuracy, GradCAM, overlay_gradcam

print(f'Device: {DEVICE}')

## 📊 Parte 1 — Métricas Finales (Test Set Completo)


In [ ]:
model_path = WEIGHTS_DIR / 'model_v1.pt'
assert model_path.exists(), 'Primero correr 03_finetuning.ipynb'

model = load_model(model_path, backbone='efficientnet_b0', device=DEVICE)

test_dataset = datasets.Food101(str(DATA_DIR), split='test', transform=get_val_transform(), download=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Test set: {len(test_dataset):,} imágenes')

In [ ]:
# Inferencia completa sobre el test set
all_preds, all_probs, all_labels = [], [], []

model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Evaluando test set'):
        images = images.to(DEVICE)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)
        
        top_probs, top_preds = probs.topk(5, dim=1)
        all_preds.append(top_preds.cpu())
        all_probs.append(top_probs[:, 0].cpu())
        all_labels.append(labels)

all_preds  = torch.cat(all_preds)
all_probs  = torch.cat(all_probs)
all_labels = torch.cat(all_labels)

top1 = (all_preds[:, 0] == all_labels).float().mean().item()
top5 = all_preds.eq(all_labels.unsqueeze(1).expand_as(all_preds)).any(1).float().mean().item()

print(f'\n=== RESULTADOS FINALES ===')
print(f'Top-1 Accuracy: {top1*100:.2f}%  (meta: >80%  {"✓" if top1 > 0.80 else "✗"})')
print(f'Top-5 Accuracy: {top5*100:.2f}%  (meta: >92%  {"✓" if top5 > 0.92 else "✗"})')

## 🗂️ Parte 2 — Matriz de Confusión y Análisis de Errores


In [ ]:
# Matriz de confusión 101×101
cm = confusion_matrix(all_labels.numpy(), all_preds[:, 0].numpy())

fig = plot_confusion_matrix(cm, FOOD101_CLASSES)
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

# Clases con mayor error
class_acc = per_class_accuracy(cm)
worst_idx = np.argsort(class_acc)[:20]

print('\n=== 20 clases con menor accuracy ===')
for idx in worst_idx:
    print(f'  {FOOD101_CLASSES[idx]:30s}  {class_acc[idx]*100:.1f}%')

## 📉 Parte 3 — Accuracy vs. Umbral de Confianza

Cuando la confianza del modelo es baja, podemos elegir no predecir
y devolver 'no sé'. Esta curva muestra el trade-off cobertura/precisión.


In [ ]:
thresholds = np.linspace(0, 1, 50)
coverages, precisions = [], []

correct = (all_preds[:, 0] == all_labels)

for thr in thresholds:
    mask = all_probs >= thr
    coverage = mask.float().mean().item()
    precision = correct[mask].float().mean().item() if mask.sum() > 0 else 0.0
    coverages.append(coverage)
    precisions.append(precision)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(coverages, precisions, 'b-o', ms=4)
ax.axhline(y=0.9, color='red', linestyle='--', label='90% accuracy')
ax.set(xlabel='Cobertura (fracción de imágenes respondidas)',
       ylabel='Accuracy (Top-1)', title='Accuracy vs. Cobertura — Umbral de confianza')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ¿Qué umbral da ≥90% de accuracy?
for thr, cov, prec in zip(thresholds, coverages, precisions):
    if prec >= 0.90:
        print(f'Para accuracy ≥90%: umbral={thr:.2f}, cobertura={cov*100:.1f}%')
        break

## 👁️ Parte 4 — Grad-CAM: ¿Qué ve el modelo?

Grad-CAM muestra qué regiones de la imagen contribuyeron más a la predicción.
Es clave para el informe: permite justificar (o cuestionar) las decisiones del modelo.


In [ ]:
# Dataset sin transform para visualizar imágenes originales
raw_test = datasets.Food101(str(DATA_DIR), split='test', transform=None, download=False)

# GradCAM sobre el último bloque del backbone
# EfficientNet-B0: model.features[-1] es el último bloque MBConv
cam = GradCAM(model, model.features[-1])
transform = get_inference_transform()

# Elegir 6 imágenes interesantes: 3 correctas + 3 incorrectas
rng = np.random.default_rng(SEED)

correct_mask   = (all_preds[:, 0] == all_labels).numpy()
incorrect_mask = ~correct_mask

correct_idx   = np.where(correct_mask)[0]
incorrect_idx = np.where(incorrect_mask)[0]

selected_correct   = rng.choice(correct_idx[:500],   size=3, replace=False)
selected_incorrect = rng.choice(incorrect_idx[:500], size=3, replace=False)
selected = list(selected_correct) + list(selected_incorrect)

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Grad-CAM — Izquierda: predicciones correctas  |  Derecha: incorrectas', fontsize=12)

for col_idx, img_idx in enumerate(selected):
    img_pil, label = raw_test[img_idx]
    
    tensor = transform(img_pil).unsqueeze(0).to(DEVICE)
    heatmap = cam.generate(tensor)
    overlay = overlay_gradcam(img_pil, heatmap)
    
    pred_cls  = FOOD101_CLASSES[all_preds[img_idx, 0].item()]
    true_cls  = FOOD101_CLASSES[label]
    is_correct = pred_cls == true_cls
    color = 'green' if is_correct else 'red'
    
    axes[0, col_idx].imshow(img_pil.resize((224, 224)))
    axes[0, col_idx].set_title(f'Real: {true_cls.replace("_", " ")}', fontsize=7)
    axes[0, col_idx].axis('off')
    
    axes[1, col_idx].imshow(overlay)
    axes[1, col_idx].set_title(f'Pred: {pred_cls.replace("_", " ")}', fontsize=7, color=color)
    axes[1, col_idx].axis('off')

cam.remove_hooks()
plt.tight_layout()
plt.savefig('gradcam_examples.png', dpi=120, bbox_inches='tight')
plt.show()

## 🍽️ Parte 5 — Demo End-to-End del Pipeline


In [ ]:
pipe = FoodVisionPipeline(model_path)

# Analizar algunas imágenes del test set
demo_indices = rng.choice(len(raw_test), size=3, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, idx in zip(axes, demo_indices):
    img_pil, true_label = raw_test[idx]
    result = pipe.analyze(img_pil, top_k=3)
    
    top_pred = result['top_prediction']
    nutr = result['nutrition']['estimated_portion'] if result['nutrition'] else {}
    true_cls = FOOD101_CLASSES[true_label]
    
    ax.imshow(img_pil.resize((224, 224)))
    ax.axis('off')
    
    title = (
        f"Real: {true_cls.replace('_', ' ')}\n"
        f"Pred: {top_pred['class'].replace('_', ' ')} ({top_pred['confidence']*100:.0f}%)\n"
        f"{nutr.get('calories', '?')} kcal · "
        f"{nutr.get('protein_g', '?')}g prot · "
        f"{nutr.get('carbs_g', '?')}g carbs · "
        f"{nutr.get('fat_g', '?')}g grasas\n"
        f"(porción ~{nutr.get('portion_g', '?')}g)"
    )
    color = 'green' if top_pred['class'] == true_cls else 'red'
    ax.set_title(title, fontsize=8, color=color)

plt.suptitle('Demo pipeline — imagen → categoría → nutrición', fontsize=11)
plt.tight_layout()
plt.show()

## 🔢 Parte 6 — Error Calórico Final


In [ ]:
# Calcular MAE y MAPE del error calórico sobre el test set completo
abs_errors, pct_errors = [], []

for i in range(len(all_labels)):
    true_cls = FOOD101_CLASSES[all_labels[i].item()]
    pred_cls = FOOD101_CLASSES[all_preds[i, 0].item()]
    
    true_nut = get_nutrition(true_cls)
    pred_nut = get_nutrition(pred_cls)
    
    if true_nut and pred_nut:
        true_kcal = true_nut['calories']
        pred_kcal = pred_nut['calories']
        abs_errors.append(abs(pred_kcal - true_kcal))
        pct_errors.append(abs(pred_kcal - true_kcal) / (true_kcal + 1e-8))

mae  = np.mean(abs_errors)
mape = np.mean(pct_errors) * 100

print(f'Error calórico MAE:  {mae:.1f} kcal/100g')
print(f'Error calórico MAPE: {mape:.1f}%  (meta: <20%  {"✓" if mape < 20 else "✗"})')

# Histograma del error porcentual
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.array(pct_errors)*100, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(mape, color='red', linestyle='--', label=f'MAPE = {mape:.1f}%')
ax.axvline(20,   color='orange', linestyle=':', label='Meta: <20%')
ax.set(xlabel='Error calórico (%)', ylabel='Frecuencia', title='Distribución del error calórico')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📋 Resumen de Resultados


In [ ]:
print('=' * 50)
print('RESUMEN FINAL — Food Vision')
print('=' * 50)
print(f'Modelo:          EfficientNet-B0 (fine-tuning)')
print(f'Dataset:         Food-101 (25,250 imágenes de test)')
print(f'')
print(f'Top-1 Accuracy:  {top1*100:.2f}%   [meta >80%]')
print(f'Top-5 Accuracy:  {top5*100:.2f}%   [meta >92%]')
print(f'Error calórico:  {mape:.1f}%    [meta <20%]')
print('=' * 50)

---
## ❓ Preguntas de reflexión

1. **¿Qué categorías tienen mayor error calórico?** ¿Coincide con las que tienen mayor error de clasificación?
2. **¿A qué umbral de confianza** conviene cortar para un uso práctico?
3. **¿Qué muestra Grad-CAM** sobre los errores? ¿El modelo mira las partes correctas del plato?
4. **¿Cómo mejorarías el sistema?** (más datos, mejor augmentación, backbone más grande, segmentación de porciones...)
